# Homicidios en Colombia (MinDefensa): registros por hecho y conteos por municipio

Registro oficial de víctimas de homicidio intencional desde 2003, con el municipio, la fecha, la zona, el sexo de la víctima, el arma y la modalidad presunta. El cuaderno puede entregar los registros tal como vienen o **conteos por municipio y mes (o año)**, con ceros en los periodos sin homicidios, listos para modelos de conteo y de series de tiempo.

|  |  |
|---|---|
| **Entidad** | Ministerio de Defensa Nacional (Policía Nacional, SIEDCO) |
| **Conjunto** | HOMICIDIO · [`m8fd-ahd9`](https://www.datos.gov.co/d/m8fd-ahd9) |
| **Una fila es** | un registro: fecha, municipio y características del hecho; `cantidad` = número de víctimas |
| **Cobertura** | todo el país, por municipio |
| **Periodo** | 01/01/2003 a 30/09/2026 |
| **Tamaño total** | 345.195 registros (consultado el 31/10/2026) |
| **Actualización** | mensual; última carga el 21/10/2026 |
| **Requisitos** | ninguno (token de datos.gov.co opcional) |
| **Tiempo de descarga** | un departamento: segundos · todo el país desde 2003: 2 a 4 minutos |

**Cómo usar este cuaderno**

1. Lea la ficha y la tabla de variables para saber qué va a descargar.
2. Ajuste la celda **«Parámetros que usted puede ajustar»** (sección 2). Es la única celda que necesita cambiar.
3. Ejecute todo (*Entorno de ejecución → Ejecutar todas* en Colab, o *Run All* en Jupyter). La base queda en la carpeta `datos/`.

Las salidas guardadas en este cuaderno salen de una **muestra pequeña** (los valores por defecto de los parámetros), para que usted vea cómo es la base antes de descargarla completa.

## Qué trae la base

Con `AGREGAR = "evento"` cada fila es un registro. Con `"municipio_mes"` o `"municipio_anio"` cada fila es un municipio en un periodo, con el total de víctimas (`homicidios`), e incluye los municipios y periodos **sin** homicidios con valor 0.

| Columna | Qué es | Tipo |
|---|---|---|
| `fecha` | Fecha del hecho | fecha |
| `cod_dpto`, `departamento` | Código y nombre del departamento | texto |
| `cod_mpio`, `municipio` | Código DANE (5 dígitos) y nombre del municipio | texto |
| `zona` | URBANA o RURAL | categórica |
| `sexo` | Sexo de la víctima | categórica |
| `arma_medio` | Arma o medio: ARMA DE FUEGO, ARMA BLANCA / CORTOPUNZANTE, CONTUNDENTES… | categórica |
| `modalidad_presunta` | Modalidad presunta: SICARIATO, RIÑAS… | categórica |
| `cantidad` | Número de víctimas del registro | número |
| `periodo`, `homicidios` | Solo en las versiones agregadas: inicio del mes o año y total de víctimas | fecha, número |

## Ideas para un modelo de machine learning

| Tarea | Variable objetivo | Predictoras posibles |
|---|---|---|
| Regresión de conteos | `homicidios` por municipio y mes | mes, rezagos del conteo, otras bases unidas por `cod_mpio` |
| Clasificación | ¿Habrá al menos un homicidio en el municipio el mes siguiente? | conteos de los meses anteriores, zona, departamento |
| Clasificación (nivel evento) | `arma_medio` o `modalidad_presunta` | zona, sexo, mes, día de la semana, municipio |
| Pronóstico | serie mensual de un departamento | la propia serie (ARIMA, clase 2.4) |

**Cuidados antes de modelar**

- Hay **muchos ceros**: la mayoría de municipios pequeños no tienen homicidios en un mes. Use métricas y modelos para conteos (Poisson) o agregue por año.
- Para comparar municipios use **tasas por 100 mil habitantes**; la población no viene en esta base (DANE, proyecciones de población).
- Valide **en el tiempo**: entrene con los meses anteriores y evalúe en los siguientes; nunca baraje los meses.
- Los datos del último mes pueden actualizarse después de publicados.

## 1. Instalación

En Colab casi todo viene instalado; esta celda agrega lo que falte.

In [ ]:
import importlib.util, subprocess, sys

PAQUETES = [("requests", "requests"), ("pandas", "pandas"), ("pyarrow", "pyarrow"), ("matplotlib", "matplotlib"), ("openpyxl", "openpyxl")]   # (módulo, paquete de pip)
faltan = [p for m, p in PAQUETES if importlib.util.find_spec(m) is None]
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Librerías listas" + (f"; se instalaron: {', '.join(faltan)}" if faltan else ""))

## 2. Parámetros que usted puede ajustar

Con `DEPARTAMENTO = None` y `AGREGAR = "municipio_mes"` se obtiene un panel de los 1.122 municipios.

> **Token opcional.** datos.gov.co funciona sin clave, pero con un *app token* gratuito las descargas grandes fallan menos: cree una cuenta en datos.gov.co → *Developer Settings* → *Create New App Token*, y antes de ejecutar escriba `os.environ["SOCRATA_APP_TOKEN"] = "su_token"`.

In [ ]:
DEPARTAMENTO = "Meta"           # con o sin tildes. None = todo el país
FECHA_INI    = "2015-01-01"     # desde (los datos empiezan el 2003-01-01)
FECHA_FIN    = None             # hasta, p. ej. "2025-12-31". None = hasta el último dato publicado
AGREGAR      = "municipio_mes"  # "evento": un registro por fila
                                # "municipio_mes" o "municipio_anio": conteos con ceros incluidos

FORMATO = "parquet"          # "parquet" (liviano y rápido), "csv" o "excel"
BAJAR_AL_COMPUTADOR = False  # en Colab, True descarga el archivo a su computador al terminar

## 3. Funciones de apoyo

No necesita cambiarlas. Descargan por páginas, reintentan si la conexión falla, resumen la base y la guardan en el formato elegido.

In [ ]:
import os, re, time, unicodedata, requests, numpy as np, pandas as pd
from pathlib import Path
from IPython.display import display

DATOS = Path("datos"); DATOS.mkdir(exist_ok=True)   # aquí quedan los archivos descargados
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

def sin_tildes(texto):
    """'Bogotá, D.C.' -> 'BOGOTA D C': mayúsculas, sin tildes ni signos."""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode().upper()
    return " ".join(re.sub(r"[^A-Z0-9 ]", " ", t).split())

def etiqueta(*partes):
    """('saber11', 'Meta') -> 'saber11_meta' (para nombrar archivos)."""
    return "_".join(sin_tildes(p).lower().replace(" ", "_") for p in partes if p)

def entero(x, _=None):
    """12345.6 -> '12.346' (separador de miles colombiano)."""
    return f"{x:,.0f}".replace(",", ".")

def decimal(x, d=1):
    """3.14159 -> '3,1' (coma decimal)."""
    return f"{x:.{d}f}".replace(".", ",")

def ficha_rapida(df):
    """Tamaño, tipo, faltantes, valores distintos y un ejemplo de cada columna."""
    print(f"{entero(len(df))} filas × {df.shape[1]} columnas | "
          f"{decimal(df.memory_usage(deep=True).sum() / 1e6)} MB en memoria")
    tabla = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "% faltante": (df.isna().mean() * 100).round(1),
        "valores distintos": df.nunique(),
        "ejemplo": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })
    display(tabla)

def guardar(df, nombre, formato=None, bajar=None):
    """Guarda la base en datos/ como parquet, csv o excel."""
    formato = (formato or FORMATO).lower()
    if formato == "excel" and len(df) > 1_048_000:
        print("Excel admite como máximo 1.048.576 filas: se guarda en parquet.")
        formato = "parquet"
    ext = {"parquet": "parquet", "csv": "csv", "excel": "xlsx"}[formato]
    ruta = DATOS / f"{nombre}.{ext}"
    sin_zona = df.copy()
    for c in sin_zona.select_dtypes("datetimetz"):
        sin_zona[c] = sin_zona[c].dt.tz_localize(None)
    if formato == "parquet": sin_zona.to_parquet(ruta, index=False)
    elif formato == "csv":   sin_zona.to_csv(ruta, index=False, encoding="utf-8-sig")
    else:                    sin_zona.to_excel(ruta, index=False)
    print(f"Guardado: {ruta} ({entero(len(df))} filas × {df.shape[1]} columnas)")
    if (BAJAR_AL_COMPUTADOR if bajar is None else bajar):
        try:
            from google.colab import files
            files.download(str(ruta))
        except ImportError:
            print("BAJAR_AL_COMPUTADOR solo funciona en Google Colab; el archivo ya está en la carpeta datos/.")
    return ruta

# ---------- datos.gov.co (API Socrata, consultas en lenguaje SoQL) ----------
BASE = "https://www.datos.gov.co/resource/{}.json"
TOKEN = os.getenv("SOCRATA_APP_TOKEN")      # opcional: os.environ["SOCRATA_APP_TOKEN"] = "su_token"
HEADERS = {"X-App-Token": TOKEN} if TOKEN else {}

def socrata(dataset_id, select=None, where=None, group=None, order=None,
            max_rows=None, limit=50_000, timeout=300, reintentos=3, mensajes=True):
    """Descarga un conjunto de datos.gov.co por páginas, con reintentos. Devuelve un DataFrame."""
    if max_rows: limit = min(limit, max_rows)
    params = {"$limit": limit}
    if select: params["$select"] = select
    if where:  params["$where"]  = where
    if group:  params["$group"]  = group
    params["$order"] = order or (group if group else ":id")    # orden estable para paginar
    partes, offset = [], 0
    while True:
        params["$offset"] = offset
        for intento in range(reintentos):
            try:
                r = requests.get(BASE.format(dataset_id), params=params, headers=HEADERS, timeout=timeout)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                if intento == reintentos - 1:
                    raise
                print(f"  reintento {intento + 1}: {e}")
                time.sleep(5 * (intento + 1))
        lote = r.json()
        if not lote:
            break
        partes.append(pd.DataFrame(lote))
        offset += len(lote)
        if mensajes:
            print(f"  {dataset_id}: {entero(offset)} filas descargadas", end="\r")
        if len(lote) < limit or (max_rows and offset >= max_rows):
            break
        time.sleep(0.3)
    if mensajes:
        print()
    df = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()
    return df.head(max_rows) if max_rows else df

def contar(dataset_id, where=None):
    """Cuántas filas cumplen el filtro, sin descargarlas."""
    r = socrata(dataset_id, select="count(*) as n", where=where, order="n", mensajes=False)
    return int(r["n"].iloc[0]) if len(r) else 0

def lista_soql(valores):
    """['a', "b'c"] -> ('a', 'b''c') para usar con IN en SoQL."""
    return "(" + ", ".join("'" + str(v).replace("'", "''") + "'" for v in valores) + ")"

def canon(texto):
    """Nombre de departamento comparable entre fuentes (con o sin tildes, 'Bogotá D.C.', 'San Andrés…')."""
    t = sin_tildes(texto)
    if "BOGOTA" in t: return "BOGOTA"
    if "SAN ANDRES" in t: return "SAN ANDRES"
    return t.replace("DEPARTAMENTO DE ", "")

def valores_de(dataset_id, campo, texto, where=None, contiene=False):
    """Busca cómo está escrito `texto` en la columna `campo` (cada fuente lo escribe distinto)."""
    df = socrata(dataset_id, select=campo, where=where, group=campo, mensajes=False)
    if df.empty:
        return []
    todos = df[campo].dropna().unique()
    if contiene:
        hallados = [v for v in todos if sin_tildes(texto) in sin_tildes(v)]
    else:
        hallados = [v for v in todos if canon(v) == canon(texto)]
    if not hallados:
        raise ValueError(f"No encontré '{texto}' en {campo}. Valores posibles: {sorted(todos)}")
    return hallados

def cod5(s):
    """Código DANE de municipio a 5 dígitos como texto (algunas fuentes traen 8 o pierden el cero)."""
    s = pd.Series(s).astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True)
    return pd.Series(np.where(s.str.len() >= 7, s.str.zfill(8).str[:5], s.str.zfill(5)), index=s.index)

def a_numero(df, cols):
    for c in cols:
        if c in df: df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# Códigos DANE de los departamentos (para filtrar por código, que no depende de tildes)
COD_DPTO = {"ANTIOQUIA": "05", "ATLANTICO": "08", "BOGOTA": "11", "BOLIVAR": "13", "BOYACA": "15",
            "CALDAS": "17", "CAQUETA": "18", "CAUCA": "19", "CESAR": "20", "CORDOBA": "23",
            "CUNDINAMARCA": "25", "CHOCO": "27", "HUILA": "41", "LA GUAJIRA": "44", "MAGDALENA": "47",
            "META": "50", "NARINO": "52", "NORTE DE SANTANDER": "54", "QUINDIO": "63", "RISARALDA": "66",
            "SANTANDER": "68", "SUCRE": "70", "TOLIMA": "73", "VALLE DEL CAUCA": "76", "ARAUCA": "81",
            "CASANARE": "85", "PUTUMAYO": "86", "SAN ANDRES": "88", "AMAZONAS": "91", "GUAINIA": "94",
            "GUAVIARE": "95", "VAUPES": "97", "VICHADA": "99"}

def cod_dpto(nombre):
    """'Nariño' -> '52'."""
    c = canon(nombre)
    if c not in COD_DPTO:
        raise ValueError(f"No reconozco el departamento '{nombre}'. Use uno de: {sorted(COD_DPTO)}")
    return COD_DPTO[c]


def divipola():
    """Los 1.122 municipios del DANE con código, nombre y coordenadas (conjunto gdxc-w37w)."""
    d = socrata("gdxc-w37w")
    d["cod_mpio"] = cod5(d["cod_mpio"])
    d["cod_dpto"] = d["cod_dpto"].astype(str).str.zfill(2)
    for c in ["latitud", "longitud"]:
        d[c] = pd.to_numeric(d[c].astype(str).str.replace(",", "."), errors="coerce")
    return d

In [ ]:
# Estilo gráfico común del curso (paleta validada para daltonismo)
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

AZUL, NARANJA, AQUA, ROJO = '#2a78d6', '#eb6834', '#1baf7a', '#e34948'
GRIS, TINTA, TINTA2 = '#898781', '#0b0b0b', '#52514e'

mpl.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'savefig.facecolor': '#fcfcfb',
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': TINTA2, 'text.color': TINTA,
    'xtick.color': GRIS, 'ytick.color': GRIS, 'axes.grid': True, 'grid.color': '#e1e0d9',
    'grid.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'figure.dpi': 110, 'lines.linewidth': 2, 'legend.frameon': False,
})

## 4. ¿Cuántos registros cumplen el filtro?

In [ ]:
HOMI = "m8fd-ahd9"
filtros = [f"fecha_hecho >= '{FECHA_INI}T00:00:00'"]
if FECHA_FIN:
    filtros.append(f"fecha_hecho <= '{FECHA_FIN}T23:59:59'")
if DEPARTAMENTO:
    filtros.append(f"cod_depto = '{cod_dpto(DEPARTAMENTO)}'")
WHERE = " AND ".join(filtros)
print("Filtro:", WHERE)
print(f"Registros que cumplen el filtro: {entero(contar(HOMI, WHERE))}")

## 5. Descarga

In [ ]:
homi = socrata(HOMI, where=WHERE)
homi = homi.rename(columns={"cod_depto": "cod_dpto", "cod_muni": "cod_mpio", "_modalidad_presunta": "modalidad_presunta"})
homi["fecha"] = pd.to_datetime(homi["fecha_hecho"])
homi["cod_mpio"] = cod5(homi["cod_mpio"])
homi["cantidad"] = pd.to_numeric(homi["cantidad"], errors="coerce").fillna(1).astype(int)
homi = homi.drop(columns="fecha_hecho")
print(f"{entero(len(homi))} registros con {entero(homi['cantidad'].sum())} víctimas, "
      f"del {homi['fecha'].min():%d/%m/%Y} al {homi['fecha'].max():%d/%m/%Y}")

## 6. Agregar por municipio y periodo

Si `AGREGAR` no es `"evento"`, se cuentan las víctimas por municipio y periodo. Para que el modelo vea también los meses sin homicidios, se arma la rejilla completa de municipios (de DIVIPOLA) × periodos y se rellena con 0.

In [ ]:
if AGREGAR == "evento":
    base = homi
elif AGREGAR in ("municipio_mes", "municipio_anio"):
    frec = "M" if AGREGAR == "municipio_mes" else "Y"
    homi["periodo"] = homi["fecha"].dt.to_period(frec).dt.to_timestamp()
    conteo = homi.groupby(["cod_mpio", "periodo"])["cantidad"].sum()

    mpios = divipola()
    if DEPARTAMENTO:
        mpios = mpios[mpios["cod_dpto"] == cod_dpto(DEPARTAMENTO)]
    fin = pd.Timestamp(FECHA_FIN) if FECHA_FIN else homi["fecha"].max()
    periodos = pd.period_range(FECHA_INI, fin, freq=frec).to_timestamp()
    rejilla = pd.MultiIndex.from_product([mpios["cod_mpio"], periodos], names=["cod_mpio", "periodo"])
    base = (conteo.reindex(rejilla, fill_value=0).rename("homicidios").reset_index()
                  .merge(mpios[["cod_mpio", "nom_mpio", "cod_dpto", "dpto"]], on="cod_mpio", how="left"))
    print(f"{mpios.shape[0]} municipios × {len(periodos)} periodos = {entero(len(base))} filas")
else:
    raise ValueError('AGREGAR debe ser "evento", "municipio_mes" o "municipio_anio"')

## 7. Revisión de la base

In [ ]:
ficha_rapida(base)
base.head()

*(Se completa al ejecutar.)*

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
serie = homi.set_index("fecha")["cantidad"].resample("MS").sum()
a1.plot(serie.index, serie.values, color=AZUL)
a1.set_title(f"Víctimas por mes — {DEPARTAMENTO or 'país'}")
arma = homi.groupby("arma_medio")["cantidad"].sum().sort_values().tail(6)
a2.barh(arma.index, arma.values, color=AZUL)
a2.set_title("Víctimas según arma o medio")
plt.tight_layout(); plt.show()

if AGREGAR != "evento":
    pct_cero = (base["homicidios"] == 0).mean() * 100
    print(f"Periodos sin homicidios: {decimal(pct_cero)} % de las filas")

*(Se completa al ejecutar.)*

## 8. Guardar

In [ ]:
ruta = guardar(base, etiqueta("homicidios", AGREGAR, DEPARTAMENTO or "pais"))

## Siguientes pasos

1. Cargue la base en otro cuaderno: `df = pd.read_parquet("datos/homicidios_municipio_mes_meta.parquet")` (o `read_csv` / `read_excel` según el `FORMATO`).
2. Defina la **pregunta** y la **variable objetivo** (vea «Ideas para un modelo de machine learning»).
3. Revise faltantes, valores extremos y tipos de variables antes de modelar (clase 2.1, ingeniería de características).
4. Parta en entrenamiento y prueba **antes** de transformar, y compare su modelo con una **línea base** sencilla.